# 使用 Microsoft Agent Framework 的 Human-in-the-Loop 工作流程

## 🎯 學習目標

在這個筆記本中，你將學習如何使用 Microsoft Agent Framework 的 `RequestInfoExecutor` 實現 **human-in-the-loop** 工作流程。這個強大的模式允許你在 AI 工作流程中暫停以收集人類輸入，使你的代理更加互動，並讓人類掌控關鍵決策。

## 🔄 何謂 Human-in-the-Loop？

**Human-in-the-loop (HITL)** 是一種設計模式，AI 代理在繼續執行前會暫停並請求人類輸入。這對於：

- ✅ <strong>關鍵決策</strong> - 在採取重要行動前獲得人類批准
- ✅ <strong>模糊情況</strong> - 當 AI 不確定時讓人類進行澄清
- ✅ <strong>使用者偏好</strong> - 請使用者在多個選項中做出選擇
- ✅ <strong>合規與安全</strong> - 確保受監管操作有人類監督
- ✅ <strong>互動體驗</strong> - 建立能回應使用者輸入的對話型代理

## 🏗️ Microsoft Agent Framework 的運作方式

該框架為 HITL 提供三個關鍵組件：

1. **`RequestInfoExecutor`** - 一個特殊的執行器，可暫停工作流程並發出 `RequestInfoEvent`
2. **`RequestInfoMessage`** - 傳送給人類的型別化請求資料的基底類別
3. **`RequestResponse`** - 使用 `request_id` 將人類回應與原始請求關聯

**工作流程範例模式：**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 我們的範例：帶使用者確認的酒店預訂

我們將在條件工作流程上新增人類確認，<strong>在</strong> 建議替代目的地之前：

1. 使用者請求一個目的地（例如「巴黎」）
2. `availability_agent` 檢查是否有空房
3. <strong>若無空房</strong> → `confirmation_agent` 詢問「您想看替代方案嗎？」
4. 使用 `RequestInfoExecutor` <strong>暫停</strong> 工作流程
5. <strong>人類透過控制台輸入</strong> 回應「是」或「否」
6. `decision_manager` 根據回應進行路由：
   - <strong>是</strong> → 顯示替代目的地
   - <strong>否</strong> → 取消預訂請求
7. 顯示最終結果

此範例示範如何讓使用者掌控代理的建議！

---

開始動手吧！🚀


## 步驟 1：導入所需庫

我們導入標準的 Agent Framework 元件以及 <strong>人機互動特定類別</strong>：
- `RequestInfoExecutor` - 會暫停工作流程以等待人類輸入的執行器
- `RequestInfoEvent` - 當請求人類輸入時所發出事件
- `RequestInfoMessage` - 用於型別請求有效負載的基底類別
- `RequestResponse` - 將人類回應與請求相關聯
- `WorkflowOutputEvent` - 用於檢測工作流程輸出的事件


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## 步驟 2：定義 Pydantic 模型以構建結構化輸出

這些模型定義了代理將返回的<strong>結構</strong>。我們保留條件工作流中的所有模型，並新增：

**人機協作環節新增：**
- `HumanFeedbackRequest` - `RequestInfoMessage` 的子類，定義發送給人的請求負載
  - 包含 `prompt`（要詢問的問題）和 `destination`（關於不可用城市的上下文）


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## 第 3 步：建立酒店預訂工具

與條件工作流程中相同的工具 – 檢查目的地是否有空房。


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## 第4步：定義路由的條件函數

我們需要<strong>四個條件函數</strong>來處理我們的人類介入工作流程：

**來自條件工作流程：**
1. `has_availability_condition` - 當有酒店可用時路由
2. `no_availability_condition` - 當酒店不可用時路由

**人類介入的新條件：**
3. `user_wants_alternatives_condition` - 當用戶說「是」要替代方案時路由
4. `user_declines_alternatives_condition` - 當用戶說「否」不需要替代方案時路由


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## 第5步：建立Decision Manager執行器

這是<strong>人機互動模式的核心</strong>！`DecisionManager` 是一個自訂的 `Executor`，它：

1. **透過 `RequestResponse` 物件接收人工回饋**
2. <strong>處理使用者的決策</strong>（是/否）
3. <strong>透過發送訊息給適當的代理來導引工作流程</strong>

主要特色：
- 使用 `@handler` 裝飾器將方法暴露為工作流程步驟
- 接收包含原始請求和使用者答案的 `RequestResponse[HumanFeedbackRequest, str]`
- 產生簡單的「是」或「否」訊息，觸發我們的條件函數


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## 步驟 6：建立自訂顯示執行器

與條件工作流程相同的顯示執行器 - 將最終結果產出為工作流程輸出。


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## 第 7 步：載入環境變數

配置 LLM 用戶端（Microsoft Foundry / Azure OpenAI）。


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## 第 8 步：建立 AI 代理與執行器

我們建立 <strong>六個工作流元件</strong>：

**代理（封裝於 AgentExecutor 中）：**
1. **availability_agent** - 使用工具檢查酒店空房情況
2. **confirmation_agent** - 🆕 準備人工確認請求
3. **alternative_agent** - 提議替代城市（當用戶表示同意時）
4. **booking_agent** - 鼓勵訂房（當有空房時）
5. **cancellation_agent** - 🆕 處理取消訊息（當用戶表示不同意時）

**特殊執行器：**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`，會暫停工作流等待人工輸入
7. **decision_manager** - 🆕 自訂執行器，根據人工回應進行路由（已於上方定義）


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## 第 9 步：建立含有人類介入的工作流程

現在我們構建包含<strong>條件路由</strong>及人類介入路徑的工作流程圖：

**工作流程結構：**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**主要邊：**
- `availability_agent → confirmation_agent`（當沒有房間時）
- `confirmation_agent → prepare_human_request`（轉換類型）
- `prepare_human_request → request_info_executor`（暫停等待人類）
- `request_info_executor → decision_manager`（始終 - 提供 RequestResponse）
- `decision_manager → alternative_agent`（當用戶說「是」時）
- `decision_manager → cancellation_agent`（當用戶說「否」時）
- `availability_agent → booking_agent`（當有房間時）
- 所有路徑結束於 `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## 步驟 10：執行測試案例 1 - 無可用性城市（巴黎需人工確認）

此測試展示了<strong>完整的人機互動循環</strong>：

1. 請求巴黎的酒店
2. availability_agent 檢查 → 無房間
3. confirmation_agent 創建面向人的問題
4. request_info_executor <strong>暫停工作流程</strong> 並發出 `RequestInfoEvent`
5. <strong>應用程式偵測事件並在控制台提示用戶</strong>
6. 使用者輸入「yes」或「no」
7. 應用程式透過 `send_responses_streaming()` 傳送回應
8. decision_manager 根據回應進行路由
9. 顯示最終結果

**主要模式：**
- 第一次迭代使用 `workflow.run_stream()`
- 隨後迭代使用 `workflow.send_responses_streaming(pending_responses)`
- 監聽 `RequestInfoEvent` 以偵測何時需人類輸入
- 監聽 `WorkflowOutputEvent` 以擷取最終結果


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## 第 11 步：執行測試案例 2 - 有房間可用的城市（斯德哥爾摩 - 無需人工輸入）

此測試演示當有房間可用時的<strong>直接路徑</strong>：

1. 請求斯德哥爾摩的酒店
2. availability_agent 檢查 → 有房間可用 ✅
3. booking_agent 建議預訂
4. display_result 顯示確認
5. **無需人工輸入！**

當有房間可用時，工作流程完全繞過人工參與的環節。


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## 重要重點及人機互動最佳實踐

### ✅ 你學到了什麼：

#### 1. **RequestInfoExecutor 範式**
Microsoft Agent Framework 中的人機互動範式使用三個主要組件：
- `RequestInfoExecutor` - 暫停工作流程並發出事件
- `RequestInfoMessage` - 類型化請求載荷的基類（請繼承此類！）
- `RequestResponse` - 將人類回應與原始請求相關聯

**關鍵理解：**
- `RequestInfoExecutor` 本身不收集輸入 - 它只暫停工作流程
- 您的應用程式程式碼必須監聽 `RequestInfoEvent` 並收集輸入
- 您必須調用 `send_responses_streaming()`，並使用映射 `request_id` 到用戶答案的字典

#### 2. <strong>串流執行範式</strong>
```python
# 第一次迭代
stream = workflow.run_stream(initial_request)

# 後續迭代（在人類輸入之後）
stream = workflow.send_responses_streaming(pending_responses)

# 始終處理事件
events = [event async for event in stream]
```

#### 3. <strong>事件驅動架構</strong>
監聽特定事件以控制工作流程：
- `RequestInfoEvent` - 需要人類輸入（工作流程暫停）
- `WorkflowOutputEvent` - 最終結果可用（工作流程完成）
- `WorkflowStatusEvent` - 狀態改變（IN_PROGRESS、IDLE_WITH_PENDING_REQUESTS 等）

#### 4. **使用 @handler 的自訂 Executor**
`DecisionManager` 展示了如何創建 executor：
- 使用 `@handler` 裝飾器將方法暴露為工作流程步驟
- 接收類型化訊息（例如，`RequestResponse[HumanFeedbackRequest, str]`）
- 透過發送訊息到其他 executor 來路由工作流程
- 通過 `WorkflowContext` 訪問上下文

#### 5. <strong>帶有人類決策的條件路由</strong>
你可以建立評估人類回應的條件函數：
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 實務應用：

1. <strong>審批工作流程</strong>
   - 在處理費用報告前獲得經理批准
   - 發送自動郵件前需人工審核
   - 執行前確認高價值交易

2. <strong>內容審查</strong>
   - 標記可疑內容供人類審核
   - 邀請審查員對疑難案件做最終決定
   - AI 信心不足時升級至人工

3. <strong>客戶服務</strong>
   - 讓 AI 自動處理常見問題
   - 將複雜問題升級給人工客服
   - 詢問客戶是否希望與人類對話

4. <strong>資料處理</strong>
   - 請人類解決模棱兩可的資料條目
   - 確認 AI 對不明確文件的解讀
   - 讓使用者在多種有效解釋間做選擇

5. <strong>安全關鍵系統</strong>
   - 不可逆行動前需人工確認
   - 訪問敏感資料前需批准
   - 在受管制產業（醫療、金融）確認決策

6. <strong>互動型代理</strong>
   - 建立會提出後續問題的對話型機器人
   - 創建引導使用者完成複雜流程的嚮導
   - 設計與人類逐步協作的代理

### 🔄 比較：有無人機互動

| 功能 | 有條件工作流程 | 人機互動工作流程 |
|---------|---------------------|---------------------------|
| <strong>執行方式</strong> | 單次 `workflow.run()` | 使用 `run_stream()` 與 `send_responses_streaming()` 迴圈 |
| <strong>用戶輸入</strong> | 無（完全自動） | 透過 `input()` 或 UI 的互動提示 |
| <strong>組件</strong> | Agents + Executors | 加上 RequestInfoExecutor + DecisionManager |
| <strong>事件</strong> | 僅有 AgentExecutorResponse | 包含 RequestInfoEvent、WorkflowOutputEvent 等 |
| <strong>暫停</strong> | 不暫停 | 工作流程在 RequestInfoExecutor 處暫停 |
| <strong>人類控制</strong> | 無人類控制 | 由人類做關鍵決策 |
| <strong>使用案例</strong> | 自動化 | 協作與監督 |

### 🚀 進階範式：

#### 多重人類決策點
同一工作流程中可以包含多個 `RequestInfoExecutor` 節點：
```python
.add_edge(agent1, request_info_1)  # 第一個人類決定
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # 第二個人類決定
.add_edge(decision_manager_2, final_agent)
```

#### 超時處理
實作人類回應的超時機制：
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # 默認為安全選項
```

#### 豐富的 UI 整合
不使用 `input()`，改與網頁 UI、Slack、Teams 等整合：
```python
if isinstance(event, RequestInfoEvent):
    # 發送通知到用戶偏好的頻道
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### 有條件的人機互動
僅在人類輸入特定情況下才請求：
```python
def needs_human_approval_condition(message: Any) -> bool:
    # 只有在信心不足或數值較高時才交由人手處理
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ 最佳實踐：

1. **始終繼承 RequestInfoMessage**
   - 提供型別安全和驗證
   - 使 UI 呈現能取得豐富上下文
   - 清晰表示每種請求的意圖

2. <strong>使用具描述性的提示</strong>
   - 包含你所詢問內容的上下文
   - 說明每種選項的結果
   - 保持問題簡單明確

3. <strong>處理意外輸入</strong>
   - 驗證用戶回應
   - 為無效輸入提供預設值
   - 給出清晰錯誤訊息

4. **追蹤 Request IDs**
   - 利用 request_id 與回應的關聯性
   - 不要嘗試手動管理狀態

5. <strong>設計非阻塞流程</strong>
   - 不要阻塞線程等待輸入
   - 全程使用非同步範式
   - 支援並行工作流程實例

### 📚 相關概念：

- **Agent Middleware** - 攔截 agent 調用（前一筆記本）
- **Workflow State Management** - 在運行間持續記錄工作流程狀態
- **Multi-Agent Collaboration** - 將人機互動與 agent 團隊結合
- **Event-Driven Architectures** - 使用事件建構反應式系統

---

### 🎓 恭喜！

你已精通 Microsoft Agent Framework 的人機互動工作流程！你現在知道如何：
- ✅ 暫停工作流程以收集人類輸入
- ✅ 使用 RequestInfoExecutor 和 RequestInfoMessage
- ✅ 透過事件處理串流執行
- ✅ 使用 @handler 創建自訂 executor
- ✅ 根據人類決策路由工作流程
- ✅ 構建與人類協作的互動式 AI 代理

**這是一個建構值得信賴且可控 AI 系統的關鍵範式！** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
